In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(1, str("../src"))
from utils.dataset import load_dataset
from utils.gpa import gpagen, two_d_array

In [2]:
specimens, meta_df = load_dataset("../data/Bombus", split="train", exclude_outliers=True, labeled_only=True)

Exclus 23 photo(s) SUSPECT/FAILED via --exclude-outliers
Filtré sur labeled_only=True : 2814 photo(s) restante(s)
Filtré sur split=train : 2571 photo(s) restante(s)


In [3]:
levels = ["species", "caste", "specimen_id", "device"]
min_top_level_n = 5
n_perm = 0

In [4]:
valid = meta_df[list(levels)].notna().all(axis=1).tolist()
n_dropped = sum(not v for v in valid)
if n_dropped:
    print(f"{n_dropped} photo(s) écartée(s) : valeur manquante sur un des niveaux {levels}")

specimens = [sp for sp, keep in zip(specimens, valid) if keep]
meta_df = meta_df[valid].reset_index(drop=True)

In [5]:
top_level = levels[0]
counts = meta_df[top_level].value_counts()
small_groups = counts[counts < min_top_level_n].index.tolist()
if small_groups:
    print(f"{top_level} écarté(s) (< {min_top_level_n} spécimens) : {small_groups}")
    mask = (~meta_df[top_level].isin(small_groups)).tolist()
    specimens = [sp for sp, keep in zip(specimens, mask) if keep]
    meta_df = meta_df[mask].reset_index(drop=True)

if len(specimens) < 3:
        raise SystemExit(f"Seulement {len(specimens)} spécimen(s) après filtrage -- pas assez pour une ANOVA.")

In [6]:
gpa_result = gpagen([sp.landmarks for sp in specimens])
X = two_d_array(gpa_result.aligned)

In [7]:
rng = np.random.default_rng(42) if n_perm else None
lvls = [(name, meta_df[name]) for name in levels]
names = [name for name, _ in lvls]

level_codes, level_sizes = [], []
for name, series in lvls:
    codes, uniques = pd.factorize(series, sort=True)
    level_codes.append(codes)
    level_sizes.append(len(uniques))

In [26]:
grand_mean = X.mean(axis=0)
ss_total = float(((X - grand_mean) ** 2).sum())
ss_total

3.7225343291443567

In [27]:
cells = []
dense, n_dense = None, 1

for codes, n in zip(level_codes, level_sizes):
    if dense is None:
        combined = codes
    else:
        combined = dense.astype(np.int64) * n + codes.astype(np.int64)
    _, dense = np.unique(combined, return_inverse=True)
    n_dense = int(dense.max()) + 1
    cells.append((dense, n_dense))
    
cells

[(array([8, 2, 7, ..., 1, 1, 4], shape=(2571,)), 12),
 (array([22,  7, 20, ...,  3,  4, 11], shape=(2571,)), 34),
 (array([340, 119, 330, ...,  60,  64, 168], shape=(2571,)), 520),
 (array([677, 238, 660, ..., 121, 129, 337], shape=(2571,)), 1035)]

In [30]:
rows = []
ss_prev, n_prev = 0.0, 1

for name, (dense, n_present) in zip(names, cells):
    group_sums = np.zeros((n_present, X.shape[1]))
    np.add.at(group_sums, dense, X)
    counts = np.bincount(dense, minlength=n_present).astype(float)
    safe_counts = np.where(counts == 0, 1.0, counts)
    group_means = group_sums / safe_counts[:, None]

    # ss_between
    ss_cells = float((counts * ((group_means - grand_mean) ** 2).sum(axis=1)).sum())
    # ss_within
    ss_level = ss_cells - ss_prev
    df_level = n_present - n_prev

    ss_prev, n_prev = ss_cells, n_present

    ms_level = ss_level / df_level if df_level > 0 else float("nan")

    print(name, ss_level, df_level, ms_level)

ss_res = ss_total - ss_prev
df_res = len(X) - n_prev
ms_res = ss_res / df_res if df_res > 0 else float("nan")
print("residuel", ss_res, df_res, ms_res)

species 1.7972190210250585 11 0.1633835473659144
caste 0.3619395257010847 22 0.016451796622776575
specimen_id 1.2214743840826943 486 0.0025133217779479307
device 0.12690813658853628 515 0.0002464235661913326
residuel 0.21499326174698297 1536 0.0001399695714498587
